<a href="https://colab.research.google.com/github/lanaajs/Processamento-de-Linguagem-Natural-Python/blob/main/Roteador_%C3%81gil_(Gest%C3%A3o_de_Backlog).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers pandas torch

In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import torch

In [ ]:
# carregando o modelo
nome_do_modelo = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
modelo_bert = SentenceTransformer(nome_do_modelo)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# separando as equipes por categorias
categorias = {
    "Equipe de Frontend e Interface":
        "interface visual botões cores layout responsivo componente react html css experiência do usuário ux ui tela navegação",

    "Equipe de Segurança da Informação":
        "segurança senha criptografia hackeado invasão autenticação proteção dados vulnerabilidade login seguro",

    "Equipe de Banco de Dados e Infraestrutura":
        "banco de dados sql server queries infraestrutura nuvem cloud aws devops performance conexão storage backup tabelas"
}

In [ ]:
frases_teste = [
    # --- Segurança ---
    "Senha deve exigir caracteres especiais.",
    "Implementar autenticação em dois fatores.",
    "Detectada tentativa de script malicioso.",
    "Não é possível entrar no servidor com a chave mestra.",
    "O sistema detectou um porta aberta",

    # --- Frontend ---
    "Botão enviar desalinhado.",
    "Menu lateral deve ser colapsável.",
    "Cores do dashboard fora do padrão.",
    "Imagens da home carregam lentamente.",
    "O usuário clicou no botão do banco de imagens.",

    # --- Banco de Dados / Infra ---
    "Consulta SQL muito lenta.",
    "Aumentar armazenamento do servidor.",
    "Falha na migração do banco.",
    "Backup noturno não executado.",
    "O engenheiro configurou o banco principal da praça."
]

In [ ]:
# transformar categoria em vetor
equipe_categoria = list(categorias.keys())
descricao_categoria = modelo_bert.encode(list(categorias.values()), convert_to_tensor=True)

In [ ]:
# pega as frases de teste
resultado = []

for frase in frases_teste:
  # transforma a frase em vetor
  frase_vetor = modelo_bert.encode(frase, convert_to_tensor=True)

  # calcula a similaridade de cosseno contra as categorias
  similaridade = util.cos_sim(frase_vetor, descricao_categoria)[0]

  # pega a maior similaridade
  indice_vencedor = torch.argmax(similaridade).item()
  score_vencedor = similaridade[indice_vencedor].item()
  equipe_vencedora = equipe_categoria[indice_vencedor]

  # armazena o resultado
  resultado.append({
    "User Story": frase,
    "Equipe Destino": equipe_vencedora,
    "Score": f"{score_vencedor:.2%}"
  })


In [ ]:
# imprime resultado
resultado_df = pd.DataFrame(resultado)
print("--- RELATÓRIO DO BACKLOG")
print(resultado_df.to_string(index=False))

--- RELATÓRIO DO BACKLOG
                                           User Story                            Equipe Destino  Score
              Senha deve exigir caracteres especiais.         Equipe de Segurança da Informação 17.13%
            Implementar autenticação em dois fatores.         Equipe de Segurança da Informação 48.98%
             Detectada tentativa de script malicioso.         Equipe de Segurança da Informação 36.65%
Não é possível entrar no servidor com a chave mestra.         Equipe de Segurança da Informação 39.89%
                   O sistema detectou um porta aberta         Equipe de Segurança da Informação 20.91%
                            Botão enviar desalinhado.         Equipe de Segurança da Informação 11.00%
                    Menu lateral deve ser colapsável. Equipe de Banco de Dados e Infraestrutura 12.47%
                   Cores do dashboard fora do padrão.            Equipe de Frontend e Interface 27.18%
                 Imagens da home carregam lentam